# Thesis Training Runner — Kaggle Kernel

This notebook orchestrates all training runs for the thesis on deep learning for CLV prediction.
Replicating Valendin et al. (2022) using CDNOW dataset.

## Setup

Install dependencies and check environment.

In [ ]:
# Install dependencies
!pip install -q torch pytorch-lightning numpy pandas scikit-learn lifetimes pyyaml tqdm seaborn wandb shap

In [ ]:
# Verify imports and configure Kaggle paths
import os
import torch
import numpy as np
import pandas as pd
from pathlib import Path

ON_KAGGLE = Path('/kaggle').exists()
DATA_ROOT = Path('/kaggle/input' if ON_KAGGLE else 'data/raw')
RESULTS_DIR = Path('/kaggle/working/results' if ON_KAGGLE else 'results')

if ON_KAGGLE:
    os.environ['KAGGLE_ENV'] = '1'
    os.environ['KAGGLE_DATA_ROOT'] = str(DATA_ROOT)
os.environ['RESULTS_DIR'] = str(RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Working directory: {Path.cwd()}")
print(f"Kaggle mode: {ON_KAGGLE}")
print(f"Data root: {DATA_ROOT}")
print(f"Results dir: {RESULTS_DIR}")
print(f"CDNOW files: {sorted(p.name for p in (DATA_ROOT / 'cdnow-dataset').glob('*')) if (DATA_ROOT / 'cdnow-dataset').exists() else 'missing cdnow-dataset'}")

## Stage 1: Valendin Replication (Base LSTM on CDNOW)

Reproduce the Base LSTM from Valendin et al. (2022) on the full CDNOW dataset.
- `lstm_base_cdnow_replication`: Base config replicating the paper
- `lstm_base_cdnow_replication_paper_finetune`: Paper's finetuning variant
- Seeds: 42, 7, 2024 (3-seed ensemble for robustness)

In [ ]:
!python -u run_seeds.py --configs lstm_base_cdnow_replication lstm_base_cdnow_replication_paper_finetune --seeds 42 7 2024 --modes sample --skip_existing --logs_dir "$RESULTS_DIR/logs"

### Generate Replication Report

Build array ensembles and generate replication diagnostics.

In [ ]:
!python -m src.evaluation.replication_report --results-dir "$RESULTS_DIR" --build-array-ensembles
# replication_report writes the interpretation files to repo-relative results/;
# mirror them into the Kaggle output directory next to the run artifacts.
!mkdir -p "$RESULTS_DIR/tables"
!cp -f results/tables/valendin_replication_interpretation.* "$RESULTS_DIR/tables/" 2>/dev/null || true

## Stage 2: Multi-Dataset Evaluation

Once Stage 1 completes, run additional configs below.

In [ ]:
# Placeholder for multi-dataset runs
# !python run_seeds.py --configs lstm_joint_cdnow lstm_joint_dunnhumby transformer_joint_cdnow --seeds 42 7 2024 --modes sample --skip_existing

## Results and Comparison

View comparison tables and generate thesis plots.

In [ ]:
# Load and display final replication outputs
tables_dir = Path(os.environ.get('RESULTS_DIR', 'results')) / 'tables'
if tables_dir.exists():
    result_files = sorted(tables_dir.glob('*'))
    print(f"Found {len(result_files)} files in {tables_dir}")
    for f in result_files[:12]:
        print(f"  - {f.name}")

    interpretation = tables_dir / 'valendin_replication_interpretation.csv'
    if interpretation.exists():
        display(pd.read_csv(interpretation))
    else:
        print('Interpretation table not found yet. Run the report cell above after all seeds finish.')
else:
    print("No results directory found yet.")